# Transformer Encoder and Decoder

## Start with the problem: transform a source into a target

Tasks such as translation map an input sequence to an output sequence. An encoder builds contextual source representations; a decoder generates target representations while consulting the encoded source. Attention connects the two streams.

This lesson distinguishes self-attention from cross-attention and traces the masks that define what each side may observe.

## See how the source and prefix meet

```text
source tokens -> encoder -> source representations --+
                                                     +-> cross-attention -> decoder output
target prefix -> masked decoder self-attention -----+
```

For a source such as `turn left`, the decoder state predicting the next target token forms a query; encoder outputs supply keys and values. Cross-attention can give more weight to the source representation for `left` when that word is being generated. For target `[y1, y2, y3]`, a shifted training pair is decoder input `[BOS, y1, y2]` and labels `[y1, y2, y3]`; the causal mask still prevents future target leakage.

## The encoder contextualizes the source

An encoder stack repeatedly applies self-attention and feed-forward transformations to source tokens. In a bidirectional encoder, a source position may attend to other non-padding source positions on either side. Padding masks prevent artificial batch elements from contributing.

The output is a sequence of contextual vectors, not a single translation. A decoder can query this sequence at every generation step.

## The decoder combines prefix and source information

A decoder layer commonly applies masked self-attention over target-prefix positions, then cross-attention whose queries come from the decoder while keys and values come from encoder outputs, followed by a feed-forward transformation. The causal mask prevents target position $t$ from seeing later target tokens; the source mask excludes padding but usually allows access to the full source.

At inference, the decoder predicts a next token, appends or selects it, and repeats. Training commonly supplies shifted target tokens (teacher forcing) so the model predicts the next token at each position in parallel.

## Align shifted targets and masks

For target tokens $[y_1,y_2,y_3]$, decoder inputs may be `[BOS, y1, y2]` and labels `[y1, y2, y3]`. The causal mask and this one-position shift jointly define next-token learning. Accidentally exposing a target token to its own prediction leaks the answer.

Encoder-decoder models are useful when outputs condition on an explicit source, such as translation or structured transformation. They are not the only architecture for these tasks; decoder-only prompting and encoder-only representations may fit other interfaces.

**Takeaway:** the encoder represents the source, masked decoder self-attention represents the target prefix, and cross-attention reads the source. Separate masks and careful target shifting preserve the intended information flow.